<a href="https://colab.research.google.com/github/EugeneOny1/-civil-infrastructure-monitoring---CS-PROJECT-II/blob/feature-branch/05_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Notebook 05 — SSD-MobileNetV2 Model Training

This notebook fine-tunes SSD-MobileNetV2 using the TensorFlow Object Detection API.

**Run this notebook in a Colab GPU runtime.**

Recommended:
`Runtime → Change runtime type → T4 GPU` when available.

Training artifacts should remain outside Git unless intentionally documented

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ---------------------------------------------------------
# 0. Host runtime check
# ---------------------------------------------------------

import os
import sys
import shutil
import subprocess
import tarfile
import urllib.request
from pathlib import Path

print("Host Python:", sys.version)

# Confirm Colab actually attached a GPU before installing anything.
gpu_check = subprocess.run(
    ["nvidia-smi"],
    text=True,
    capture_output=True
)

if gpu_check.returncode != 0:
    raise RuntimeError(
        "No NVIDIA GPU is attached to this Colab runtime. "
        "Go to Runtime -> Change runtime type and select A100/L4/T4 GPU."
    )

print("NVIDIA GPU detected by host runtime.")



Host Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
NVIDIA GPU detected by host runtime.


In [ ]:
import sys
import platform

print("Python:", sys.version)
print("Platform:", platform.platform())

try:
    import tensorflow as tf
    print("Preinstalled TensorFlow:", tf.__version__)
except Exception as e:
    print("TensorFlow not currently importable:", e)

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
Preinstalled TensorFlow: 2.21.0


## 1. Create an Isolated TensorFlow 2.15 Environment
The TensorFlow Object Detection API still relies on TensorFlow Estimator.
To avoid changing Colab's own packages, all TensorFlow/Object Detection dependencies are installed in an isolated environment.
TensorFlow 2.15.0.post1 is used because it is the official Linux hot-fix for the TensorFlow 2.15 and-cuda installation.

In [ ]:
# ---------------------------------------------------------
# Create clean isolated Python environment
# ---------------------------------------------------------

TFOD_ENV = Path("/content/tfod_env")

if TFOD_ENV.exists():
    shutil.rmtree(TFOD_ENV)

# virtualenv is used instead of python -m venv because some
# Colab runtime images have had ensurepip problems.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "virtualenv"
    ],
    check=True
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "virtualenv",
        str(TFOD_ENV)
    ],
    check=True
)

TFOD_PYTHON = str(TFOD_ENV / "bin" / "python")
TFOD_PIP = str(TFOD_ENV / "bin" / "pip")

assert Path(TFOD_PYTHON).exists(), "Isolated Python was not created."
assert Path(TFOD_PIP).exists(), "Isolated pip was not created."

print("Environment:", TFOD_ENV)
print("Python:", TFOD_PYTHON)
print("pip:", TFOD_PIP)

Environment: /content/tfod_env
Python: /content/tfod_env/bin/python
pip: /content/tfod_env/bin/pip


In [ ]:
# ---------------------------------------------------------
# Python 3.13 + TensorFlow 2.20 TFOD environment
# ---------------------------------------------------------

import subprocess
import sys

print("Notebook Python:", sys.version)

# Upgrade build tooling
subprocess.run(
    [
        TFOD_PIP,
        "install",
        "-q",
        "--upgrade",
        "pip",
        "setuptools",
        "wheel"
    ],
    check=True
)

packages = [
    "tensorflow==2.20.0",
    "tf-models-official==2.20.0",
    "tf-slim",
    "pycocotools",
    "lvis",
    "contextlib2",
    "avro-python3",
    "sacrebleu==2.2.0",
]

install_result = subprocess.run(
    [TFOD_PIP, "install", "--upgrade", *packages],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print(install_result.stdout)

if install_result.returncode != 0:
    raise RuntimeError(
        "TFOD dependency installation failed. "
        "Inspect the pip output above."
    )

print("\nPython 3.13 / TensorFlow 2.20 stack installed.")

Notebook Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build 

In [ ]:
# ---------------------------------------------------------
# 1.3A Install TensorFlow 2.20 GPU runtime libraries
# ---------------------------------------------------------

import subprocess

gpu_install = subprocess.run(
    [
        TFOD_PIP,
        "install",
        "--upgrade",
        "tensorflow[and-cuda]==2.20.0"
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print(gpu_install.stdout)

if gpu_install.returncode != 0:
    raise RuntimeError(
        "TensorFlow CUDA runtime installation failed."
    )

print(
    "\nTensorFlow 2.20 CUDA runtime dependencies installed."
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 12.3 MB/s  0:00:26
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 14.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 17.2 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 MB 13.7 MB/s  0:00:06
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 17.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 716.6/716.6 MB 13.1 MB/s  0:00:38
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 18.7 MB/s  0:00:10
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 14.8 MB/s  0:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.1/338.1 MB 12.5 MB/s  0:00:24
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.5/366.5 MB 14.0 MB/s  0:00:27
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.5/351.5 MB 16.9 MB/s  0:00:23
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.7/39.7 MB 27.6 MB/s  0:00:01



TensorFlow 2.20 CUDA runtime dependencies installed.


In [ ]:
# ---------------------------------------------------------
# 1.3B Verify TensorFlow 2.20 + GPU
# ---------------------------------------------------------

import subprocess
import os

env = os.environ.copy()
env["TF_USE_LEGACY_KERAS"] = "1"

check = subprocess.run(
    [
        TFOD_PYTHON,
        "-c",
        (
            "import sys; "
            "import tensorflow as tf; "
            "print('Python:', sys.version); "
            "print('TensorFlow:', tf.__version__); "
            "print('GPU:', tf.config.list_physical_devices('GPU')); "
            "print('Built with CUDA:', tf.test.is_built_with_cuda()); "
            "print('CUDA build info:', tf.sysconfig.get_build_info())"
        )
    ],
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print(check.stdout)

assert check.returncode == 0, (
    "TensorFlow could not be imported inside the TFOD environment."
)

assert "Python: 3.13" in check.stdout, (
    "Expected Python 3.13.x."
)

assert "TensorFlow: 2.20" in check.stdout, (
    "Expected TensorFlow 2.20.x."
)

assert "Built with CUDA: True" in check.stdout, (
    "TensorFlow was not built with CUDA."
)

assert "GPU: []" not in check.stdout, (
    "TensorFlow still cannot see the GPU."
)

print(
    "\nPython 3.13 / TensorFlow 2.20 GPU environment PASSED."
)

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Built with CUDA: True
CUDA build info: OrderedDict({'cpu_compiler': 'clang 18', 'cuda_compute_capabilities': ['sm_60', 'sm_70', 'sm_80', 'sm_89', 'compute_90'], 'cuda_version': '12.5.1', 'cudnn_version': '9', 'is_cuda_build': True, 'is_rocm_build': False, 'is_tensorrt_build': False})


Python 3.13 / TensorFlow 2.20 GPU environment PASSED.


## 1. Project Paths and Training Inputs
Notebook 04 already produced the TFRecords and label map.
This section only verifies that those files are available.

In [ ]:
# ---------------------------------------------------------
# Project paths
# ---------------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/CS PROJECT II")
DATA_DIR = PROJECT_DIR / "Data"
PROCESSED_DIR = DATA_DIR / "Processed Data"

TFRECORD_DIR = PROCESSED_DIR / "tfrecords"

LABEL_MAP_PATH = PROJECT_DIR / "configs" / "label_map.pbtxt"

TRAIN_RECORD = TFRECORD_DIR / "train.record"
VAL_RECORD = TFRECORD_DIR / "validation.record"
TEST_RECORD = TFRECORD_DIR / "test.record"

TRAINING_DIR = PROJECT_DIR / "training_checkpoints" / "ssd_mobilenet_v2_fpnlite_640_run_v3"
EXPORT_DIR = PROJECT_DIR / "exported_model_v3"

print("Train record:", TRAIN_RECORD)
print("Validation record:", VAL_RECORD)
print("Test record:", TEST_RECORD)
print("Label map:", LABEL_MAP_PATH)
print("Training directory:", TRAINING_DIR)

Train record: /content/drive/MyDrive/CS PROJECT II/Data/Processed Data/tfrecords/train.record
Validation record: /content/drive/MyDrive/CS PROJECT II/Data/Processed Data/tfrecords/validation.record
Test record: /content/drive/MyDrive/CS PROJECT II/Data/Processed Data/tfrecords/test.record
Label map: /content/drive/MyDrive/CS PROJECT II/configs/label_map.pbtxt
Training directory: /content/drive/MyDrive/CS PROJECT II/training_checkpoints/ssd_mobilenet_v2_fpnlite_640_run_v3


In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/CS PROJECT II")

print("Project dir exists:", PROJECT_DIR.exists())
print("Project dir:", PROJECT_DIR)

print("\nTop-level project contents:")
if PROJECT_DIR.exists():
    for item in sorted(PROJECT_DIR.iterdir()):
        print(" -", item.name)

Project dir exists: True
Project dir: /content/drive/MyDrive/CS PROJECT II

Top-level project contents:
 - About the MobileNet SSD v2.gdoc
 - Colab Notebooks
 - Data
 - YOLO to TFRecord Format Conversion.gdoc
 - configs
 - exported_model
 - exported_model_v2
 - report_figures
 - training_checkpoints


In [ ]:
# ---------------------------------------------------------
# Input preflight
# ---------------------------------------------------------

required_files = {
    "Train TFRecord": TRAIN_RECORD,
    "Validation TFRecord": VAL_RECORD,
    "Test TFRecord": TEST_RECORD,
    "Label map": LABEL_MAP_PATH,
}

for name, path in required_files.items():
    print(
        f"{name}:",
        "OK" if path.exists() else "MISSING",
        "->",
        path
    )

for name, path in required_files.items():
    assert path.exists(), f"{name} is missing: {path}"

print("\nTraining inputs are ready.")

Train TFRecord: OK -> /content/drive/MyDrive/CS PROJECT II/Data/Processed Data/tfrecords/train.record
Validation TFRecord: OK -> /content/drive/MyDrive/CS PROJECT II/Data/Processed Data/tfrecords/validation.record
Test TFRecord: OK -> /content/drive/MyDrive/CS PROJECT II/Data/Processed Data/tfrecords/test.record
Label map: OK -> /content/drive/MyDrive/CS PROJECT II/configs/label_map.pbtxt

Training inputs are ready.


## 2. Install TensorFlow Object Detection API

The source tree is pinned to TensorFlow Models v2.15.0, matching the TensorFlow 2.15 training environment.
The API is loaded directly from the source tree through PYTHONPATH; it is not installed into Colab's global Python environment.

In [ ]:
# ---------------------------------------------------------
# Download TensorFlow Object Detection API source
# ---------------------------------------------------------

MODELS_DIR = Path("/content/models")

# Remove any incomplete/incorrect previous clone
if MODELS_DIR.exists():
    shutil.rmtree(MODELS_DIR)

print("Cloning TensorFlow Models repository...")

clone_result = subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/tensorflow/models.git",
        str(MODELS_DIR),
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print(clone_result.stdout)

if clone_result.returncode != 0:
    raise RuntimeError(
        "TensorFlow Models repository clone failed."
    )

RESEARCH_DIR = MODELS_DIR / "research"
OBJECT_DETECTION_DIR = RESEARCH_DIR / "object_detection"
PROTOS_DIR = OBJECT_DETECTION_DIR / "protos"
SLIM_DIR = RESEARCH_DIR / "slim"

print("\nResearch directory:", RESEARCH_DIR)
print("Object Detection directory:", OBJECT_DETECTION_DIR)
print("Protos directory:", PROTOS_DIR)

assert RESEARCH_DIR.exists(), (
    f"Missing research directory: {RESEARCH_DIR}"
)

assert OBJECT_DETECTION_DIR.exists(), (
    f"Missing Object Detection directory: {OBJECT_DETECTION_DIR}"
)

assert PROTOS_DIR.exists(), (
    f"Missing protos directory: {PROTOS_DIR}"
)

proto_count = len(list(PROTOS_DIR.glob("*.proto")))

print("Proto files found:", proto_count)

assert proto_count > 0, (
    "Repository cloned, but no Object Detection proto files were found."
)

print("\nTensorFlow Object Detection source is ready.")

Cloning TensorFlow Models repository...
Cloning into '/content/models'...


Research directory: /content/models/research
Object Detection directory: /content/models/research/object_detection
Protos directory: /content/models/research/object_detection/protos
Proto files found: 34

TensorFlow Object Detection source is ready.


In [ ]:
# ---------------------------------------------------------
# Fix missing pkg_resources inside isolated TFOD environment
# ---------------------------------------------------------

subprocess.run(
    [
        TFOD_PIP,
        "install",
        "--upgrade",
        "setuptools==68.2.2"
    ],
    check=True
)

print("setuptools/pkg_resources installed.")

setuptools/pkg_resources installed.


In [ ]:
result = subprocess.run(
    [
        TFOD_PYTHON,
        "-c",
        "import pkg_resources; print('pkg_resources: OK')"
    ],
    text=True,
    capture_output=True
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

assert result.returncode == 0

pkg_resources: OK

<string>:1: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html



In [ ]:
# ---------------------------------------------------------
# Install protobuf compiler support inside TFOD environment
# ---------------------------------------------------------

import subprocess

result = subprocess.run(
    [
        TFOD_PIP,
        "install",
        "--upgrade",
        "grpcio-tools"
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print(result.stdout)

if result.returncode != 0:
    raise RuntimeError(
        "Failed to install grpcio-tools in TFOD environment."
    )

print("grpcio-tools installed successfully.")

  Using cached setuptools-84.2.1-py3-none-any.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 14.2 MB/s  0:00:00
Using cached setuptools-84.2.1-py3-none-any.whl (876 kB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 68.2.2
    Uninstalling setuptools-68.2.2:
      Successfully uninstalled setuptools-68.2.2


grpcio-tools installed successfully.


In [ ]:
check = subprocess.run(
    [
        TFOD_PYTHON,
        "-c",
        "import grpc_tools; print('grpc_tools: OK')"
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print(check.stdout)

assert check.returncode == 0

grpc_tools: OK



In [ ]:
# ---------------------------------------------------------
# Compile Object Detection protobuf definitions
# ---------------------------------------------------------

proto_files = sorted(
    PROTOS_DIR.glob("*.proto")
)

print("Proto files to compile:", len(proto_files))

assert proto_files, (
    f"No .proto files found in {PROTOS_DIR}"
)

command = [
    TFOD_PYTHON,
    "-m",
    "grpc_tools.protoc",
    f"-I{RESEARCH_DIR}",
    f"--python_out={RESEARCH_DIR}",
]

command += [
    str(p.relative_to(RESEARCH_DIR))
    for p in proto_files
]

proto_result = subprocess.run(
    command,
    cwd=str(RESEARCH_DIR),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print(proto_result.stdout)

if proto_result.returncode != 0:
    raise RuntimeError(
        "Object Detection protobuf compilation failed."
    )

TFOD_RUN_ENV = os.environ.copy()
TFOD_RUN_ENV["TF_USE_LEGACY_KERAS"] = "1"

TFOD_RUN_ENV["PYTHONPATH"] = (
    f"{RESEARCH_DIR}:{SLIM_DIR}"
)

# Prevent Colab's notebook Matplotlib backend from leaking
# into the isolated TFOD Python environment.
TFOD_RUN_ENV["MPLBACKEND"] = "Agg"

print("Protobuf compilation: OK")
print(
    "TFOD PYTHONPATH:",
    TFOD_RUN_ENV["PYTHONPATH"]
)

Proto files to compile: 34

Protobuf compilation: OK
TFOD PYTHONPATH: /content/models/research:/content/models/research/slim


In [ ]:
# ---------------------------------------------------------
# 3.2B Python 3.13 compatibility patch:
#      Make unused DeepMAC / tensorflow-io dependency optional
# ---------------------------------------------------------

from pathlib import Path

MODEL_BUILDER = (
    RESEARCH_DIR
    / "object_detection"
    / "builders"
    / "model_builder.py"
)

assert MODEL_BUILDER.exists(), (
    f"model_builder.py not found: {MODEL_BUILDER}"
)

text = MODEL_BUILDER.read_text(encoding="utf-8")

old_import = (
    "from object_detection.meta_architectures "
    "import deepmac_meta_arch"
)

replacement = """try:
  from object_detection.meta_architectures import deepmac_meta_arch
except ModuleNotFoundError as exc:
  if exc.name == 'tensorflow_io':
    deepmac_meta_arch = None
  else:
    raise
"""

if old_import in text:
    text = text.replace(
        old_import,
        replacement,
        1
    )

MODEL_BUILDER.write_text(
    text,
    encoding="utf-8"
)

patched = MODEL_BUILDER.read_text(encoding="utf-8")

assert "deepmac_meta_arch = None" in patched

print(
    "DeepMAC optional-import compatibility patch applied."
)
print(
    "This does not modify SSD-MobileNetV2 FPNLite functionality."
)

DeepMAC optional-import compatibility patch applied.
This does not modify SSD-MobileNetV2 FPNLite functionality.


In [ ]:
# ---------------------------------------------------------
# 3.3 Verify TFOD components required by V3
#     SSD-MobileNetV2 FPNLite only
# ---------------------------------------------------------

import subprocess

verification = subprocess.run(
    [
        TFOD_PYTHON,
        "-c",
        (
            "import tensorflow as tf; "
            "from object_detection.utils import config_util; "
            "from object_detection.builders import model_builder; "
            "from object_detection.utils import label_map_util; "
            "print('TensorFlow:', tf.__version__); "
            "print('GPU:', tf.config.list_physical_devices('GPU')); "
            "print('config_util: OK'); "
            "print('model_builder: OK'); "
            "print('label_map_util: OK'); "
            "print('SSD-MobileNetV2 FPNLite dependencies: OK')"
        )
    ],
    cwd=str(RESEARCH_DIR),
    env=TFOD_RUN_ENV,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print(verification.stdout)

if verification.returncode != 0:
    raise RuntimeError(
        "Required TFOD SSD components failed to import."
    )

assert "GPU: []" not in verification.stdout, (
    "GPU is not visible inside the TFOD environment."
)

print(
    "\nRequired TensorFlow Object Detection API components PASSED."
)

TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
config_util: OK
model_builder: OK
label_map_util: OK
SSD-MobileNetV2 FPNLite dependencies: OK


Required TensorFlow Object Detection API components PASSED.


## 3. Download the Pretrained SSD-MobileNetV2 Checkpoint

The checkpoint provides transfer-learning weights. The final model will be fine-tuned for the project's three classes.

In [ ]:
# ---------------------------------------------------------
# Download pretrained SSD-MobileNetV2 FPNLite 640x640
# ---------------------------------------------------------

PRETRAINED_NAME = "ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8"

PRETRAINED_URL = (
    "http://download.tensorflow.org/models/object_detection/tf2/20200711/"
    "ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8.tar.gz"
)

PRETRAINED_ROOT = Path("/content/pretrained")
PRETRAINED_ROOT.mkdir(parents=True, exist_ok=True)

archive = PRETRAINED_ROOT / f"{PRETRAINED_NAME}.tar.gz"
model_dir = PRETRAINED_ROOT / PRETRAINED_NAME

if not model_dir.exists():
    print("Downloading pretrained model...")
    urllib.request.urlretrieve(PRETRAINED_URL, archive)

    with tarfile.open(archive, "r:gz") as tar:
        tar.extractall(PRETRAINED_ROOT)

PRETRAINED_CONFIG = model_dir / "pipeline.config"
CHECKPOINT_PREFIX = model_dir / "checkpoint" / "ckpt-0"

assert PRETRAINED_CONFIG.exists(), (
    f"Missing pretrained config: {PRETRAINED_CONFIG}"
)

assert (CHECKPOINT_PREFIX.parent / "ckpt-0.index").exists(), (
    "Missing pretrained checkpoint."
)

print("Pretrained model:", PRETRAINED_NAME)
print("Pipeline config:", PRETRAINED_CONFIG)
print("Checkpoint:", CHECKPOINT_PREFIX)

Pretrained model: ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8
Pipeline config: /content/pretrained/ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8/pipeline.config
Checkpoint: /content/pretrained/ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8/checkpoint/ckpt-0


/tmp/ipykernel_272/1642630984.py:23: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(PRETRAINED_ROOT)


## 4. Prepare a Runtime Pipeline Configuration

The repository `configs/pipeline.config` is treated as a project configuration reference. This notebook creates a runtime copy rather than silently overwriting the repository file.

**If the existing pipeline config is a placeholder or uses a different model architecture, inspect it before training.**

In [ ]:
# ---------------------------------------------------------
# Create runtime pipeline configuration
# ---------------------------------------------------------

RUNTIME_CONFIG = (
    PROJECT_DIR
    / "configs"
    / "pipeline_runtime.config"
)

RUNTIME_CONFIG.parent.mkdir(
    parents=True,
    exist_ok=True
)

shutil.copy2(
    PRETRAINED_CONFIG,
    RUNTIME_CONFIG
)

print("Source config:", PRETRAINED_CONFIG)
print("Runtime config:", RUNTIME_CONFIG)

Source config: /content/pretrained/ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8/pipeline.config
Runtime config: /content/drive/MyDrive/CS PROJECT II/configs/pipeline_runtime.config


## 5. Patch the Pipeline Configuration

This uses the TensorFlow Object Detection API protobuf rather than fragile string replacement.

The model is configured for:
- 3 classes
- batch size 8 by default
- pretrained SSD-MobileNetV2 checkpoint
- train/validation TFRecords
- project label map

In [ ]:
# ---------------------------------------------------------
# V3 — Patch pipeline configuration
# SSD-MobileNetV2 FPNLite 640x640
# ---------------------------------------------------------

patch_script = f"""
from google.protobuf import text_format
from object_detection.protos import pipeline_pb2, preprocessor_pb2

runtime_config = {str(RUNTIME_CONFIG)!r}
label_map = {str(LABEL_MAP_PATH)!r}
train_record = {str(TRAIN_RECORD)!r}
val_record = {str(VAL_RECORD)!r}
checkpoint = {str(CHECKPOINT_PREFIX)!r}

pipeline_config = pipeline_pb2.TrainEvalPipelineConfig()

with open(runtime_config, "r", encoding="utf-8") as f:
    text_format.Merge(f.read(), pipeline_config)

# ---------------------------------------------------------
# Project classes
# ---------------------------------------------------------

pipeline_config.model.ssd.num_classes = 3

# ---------------------------------------------------------
# Confirm 640 x 640 model input
# ---------------------------------------------------------

pipeline_config.model.ssd.image_resizer.fixed_shape_resizer.height = 640
pipeline_config.model.ssd.image_resizer.fixed_shape_resizer.width = 640

# ---------------------------------------------------------
# Batch size
# ---------------------------------------------------------

pipeline_config.train_config.batch_size = 4

# ---------------------------------------------------------
# Transfer learning
# ---------------------------------------------------------

pipeline_config.train_config.fine_tune_checkpoint = checkpoint
pipeline_config.train_config.fine_tune_checkpoint_type = "detection"

# ---------------------------------------------------------
# Learning-rate schedule
# ---------------------------------------------------------

lr = (
    pipeline_config
    .train_config
    .optimizer
    .momentum_optimizer
    .learning_rate
    .cosine_decay_learning_rate
)

lr.learning_rate_base = 0.0025
lr.total_steps = 12000
lr.warmup_learning_rate = 0.0005
lr.warmup_steps = 600

# ---------------------------------------------------------
# Data augmentation
#
# Preserve the official FPNLite crop/flip strategy and add
# mild illumination variation suitable for field imagery.
# ---------------------------------------------------------

existing_augments = [
    step.WhichOneof("preprocessing_step")
    for step in pipeline_config.train_config.data_augmentation_options
]

print("Original augmentations:", existing_augments)

# Keep pretrained model's horizontal flip and random crop.
# Add mild brightness and contrast only if not already present.

if "random_adjust_brightness" not in existing_augments:
    step = pipeline_config.train_config.data_augmentation_options.add()
    step.random_adjust_brightness.max_delta = 0.12

if "random_adjust_contrast" not in existing_augments:
    step = pipeline_config.train_config.data_augmentation_options.add()
    step.random_adjust_contrast.min_delta = 0.85
    step.random_adjust_contrast.max_delta = 1.15

# ---------------------------------------------------------
# Dataset paths
# ---------------------------------------------------------

pipeline_config.train_input_reader.label_map_path = label_map
pipeline_config.train_input_reader.tf_record_input_reader.input_path[:] = [
    train_record
]

pipeline_config.eval_input_reader[0].label_map_path = label_map
pipeline_config.eval_input_reader[0].tf_record_input_reader.input_path[:] = [
    val_record
]

# ---------------------------------------------------------
# Save patched runtime config
# ---------------------------------------------------------

with open(runtime_config, "w", encoding="utf-8") as f:
    f.write(text_format.MessageToString(pipeline_config))

final_augments = [
    step.WhichOneof("preprocessing_step")
    for step in pipeline_config.train_config.data_augmentation_options
]

print("\\nArchitecture: SSD-MobileNetV2 FPNLite")
print("Input resolution:",
      pipeline_config.model.ssd.image_resizer.fixed_shape_resizer.height,
      "x",
      pipeline_config.model.ssd.image_resizer.fixed_shape_resizer.width)

print("Classes:", pipeline_config.model.ssd.num_classes)
print("Batch size:", pipeline_config.train_config.batch_size)
print("Base LR:", lr.learning_rate_base)
print("Warmup LR:", lr.warmup_learning_rate)
print("Warmup steps:", lr.warmup_steps)
print("Maximum training steps:", lr.total_steps)
print("Fine-tune checkpoint:", pipeline_config.train_config.fine_tune_checkpoint)
print("Augmentations:", final_augments)

print(
    "Train record:",
    pipeline_config.train_input_reader
    .tf_record_input_reader.input_path[0]
)

print(
    "Validation record:",
    pipeline_config.eval_input_reader[0]
    .tf_record_input_reader.input_path[0]
)
"""

patch_result = subprocess.run(
    [TFOD_PYTHON, "-c", patch_script],
    env=TFOD_RUN_ENV,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print(patch_result.stdout)

if patch_result.returncode != 0:
    raise RuntimeError(
        "V3 pipeline configuration patch failed."
    )

assert RUNTIME_CONFIG.exists()

print("\nV3 runtime pipeline configuration READY.")

Original augmentations: ['random_horizontal_flip', 'random_crop_image']

Architecture: SSD-MobileNetV2 FPNLite
Input resolution: 640 x 640
Classes: 3
Batch size: 4
Base LR: 0.0024999999441206455
Warmup LR: 0.0005000000237487257
Warmup steps: 600
Maximum training steps: 12000
Fine-tune checkpoint: /content/pretrained/ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8/checkpoint/ckpt-0
Augmentations: ['random_horizontal_flip', 'random_crop_image', 'random_adjust_brightness', 'random_adjust_contrast']
Train record: /content/drive/MyDrive/CS PROJECT II/Data/Processed Data/tfrecords/train.record
Validation record: /content/drive/MyDrive/CS PROJECT II/Data/Processed Data/tfrecords/validation.record


V3 runtime pipeline configuration READY.


In [ ]:
# ---------------------------------------------------------
# Verify V3 training configuration
# ---------------------------------------------------------

verify_hparams = f"""
from google.protobuf import text_format
from object_detection.protos import pipeline_pb2

config = pipeline_pb2.TrainEvalPipelineConfig()

with open({str(RUNTIME_CONFIG)!r}, "r", encoding="utf-8") as f:
    text_format.Merge(f.read(), config)

lr = (
    config
    .train_config
    .optimizer
    .momentum_optimizer
    .learning_rate
    .cosine_decay_learning_rate
)

height = config.model.ssd.image_resizer.fixed_shape_resizer.height
width = config.model.ssd.image_resizer.fixed_shape_resizer.width

augmentations = [
    step.WhichOneof("preprocessing_step")
    for step in config.train_config.data_augmentation_options
]

architecture = config.model.ssd.feature_extractor.type
num_classes = config.model.ssd.num_classes
batch_size = config.train_config.batch_size

print("Architecture:", architecture)
print("Input size:", height, "x", width)
print("Classes:", num_classes)
print("Batch size:", batch_size)
print("Learning rate base:", f"{{lr.learning_rate_base:.6f}}")
print("Warmup learning rate:", f"{{lr.warmup_learning_rate:.6f}}")
print("Warmup steps:", lr.warmup_steps)
print("Total steps:", lr.total_steps)
print("Augmentations:", augmentations)

# ---------------------------------------------------------
# Verify actual configuration values numerically
# ---------------------------------------------------------

assert architecture == "ssd_mobilenet_v2_fpn_keras"

assert height == 640
assert width == 640

assert num_classes == 3
assert batch_size == 4

assert abs(lr.learning_rate_base - 0.0025) < 1e-7
assert abs(lr.warmup_learning_rate - 0.0005) < 1e-7

assert lr.warmup_steps == 600
assert lr.total_steps == 12000

expected_augmentations = [
    "random_horizontal_flip",
    "random_crop_image",
    "random_adjust_brightness",
    "random_adjust_contrast",
]

for aug in expected_augmentations:
    assert aug in augmentations, f"Missing augmentation: {{aug}}"

print("\\nV3 training configuration VERIFIED.")
"""

result = subprocess.run(
    [TFOD_PYTHON, "-c", verify_hparams],
    env=TFOD_RUN_ENV,
    text=True,
    capture_output=True
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

assert result.returncode == 0, (
    "V3 hyperparameter verification failed."
)

Architecture: ssd_mobilenet_v2_fpn_keras
Input size: 640 x 640
Classes: 3
Batch size: 4
Learning rate base: 0.002500
Warmup learning rate: 0.000500
Warmup steps: 600
Total steps: 12000
Augmentations: ['random_horizontal_flip', 'random_crop_image', 'random_adjust_brightness', 'random_adjust_contrast']

V3 training configuration VERIFIED.



## 6. Training

Start with a manageable number of steps and inspect the loss before committing to a long run.

**Do not assume a specific number of steps is sufficient for the final model.** The final number should be based on validation behaviour and available training time.

In [ ]:
# ---------------------------------------------------------
# TF-Slim compatibility patch for TensorFlow 2.20
# Python 3.13
# ---------------------------------------------------------

from pathlib import Path
import subprocess

# ---------------------------------------------------------
# Locate TFOD environment site-packages WITHOUT importing
# tf_slim, because tf_slim itself may currently fail to import.
# ---------------------------------------------------------

site_result = subprocess.run(
    [
        TFOD_PYTHON,
        "-c",
        "import site; print(site.getsitepackages()[0])"
    ],
    text=True,
    capture_output=True
)

print(site_result.stdout)

if site_result.stderr:
    print(site_result.stderr)

assert site_result.returncode == 0, (
    "Could not locate TFOD environment site-packages."
)

site_packages = Path(site_result.stdout.strip())

TFSLIM_DECODER = (
    site_packages
    / "tf_slim"
    / "data"
    / "tfexample_decoder.py"
)

print("TFOD site-packages:", site_packages)
print("TF-Slim decoder:", TFSLIM_DECODER)

assert TFSLIM_DECODER.exists(), (
    f"Could not find tfexample_decoder.py at: {TFSLIM_DECODER}"
)

# ---------------------------------------------------------
# Read decoder
# ---------------------------------------------------------

text = TFSLIM_DECODER.read_text(
    encoding="utf-8"
)

# ---------------------------------------------------------
# Check whether compatibility patch is needed
# ---------------------------------------------------------

needs_patch = (
    "control_flow_ops.case(" in text
    or "control_flow_ops.cond(" in text
)

print("Compatibility patch required:", needs_patch)

# ---------------------------------------------------------
# Add public TensorFlow import only if needed
# ---------------------------------------------------------

if needs_patch and "import tensorflow as tf" not in text:

    marker = "from tensorflow.python"

    if marker in text:
        text = text.replace(
            marker,
            "import tensorflow as tf\n\n" + marker,
            1
        )
    else:
        raise RuntimeError(
            "Could not safely insert TensorFlow import."
        )

# ---------------------------------------------------------
# Replace obsolete internal TensorFlow control-flow APIs
# ---------------------------------------------------------

replacements = {
    "control_flow_ops.case(": "tf.case(",
    "control_flow_ops.cond(": "tf.cond(",
}

total_replacements = 0

for old, new in replacements.items():

    count = text.count(old)

    print(f"{old} occurrences:", count)

    if count:
        text = text.replace(old, new)
        total_replacements += count

# ---------------------------------------------------------
# Save patched decoder
# ---------------------------------------------------------

TFSLIM_DECODER.write_text(
    text,
    encoding="utf-8"
)

# ---------------------------------------------------------
# Verify patch
# ---------------------------------------------------------

patched = TFSLIM_DECODER.read_text(
    encoding="utf-8"
)

assert "control_flow_ops.case(" not in patched
assert "control_flow_ops.cond(" not in patched

if (
    "tf.case(" in patched
    or "tf.cond(" in patched
):
    assert "import tensorflow as tf" in patched

print(
    f"\nTF-Slim TensorFlow 2.20 compatibility patch COMPLETE. "
    f"Replacements applied: {total_replacements}"
)

/content/tfod_env/lib/python3.13/site-packages

TFOD site-packages: /content/tfod_env/lib/python3.13/site-packages
TF-Slim decoder: /content/tfod_env/lib/python3.13/site-packages/tf_slim/data/tfexample_decoder.py
Compatibility patch required: True
control_flow_ops.case( occurrences: 1
control_flow_ops.cond( occurrences: 2

TF-Slim TensorFlow 2.20 compatibility patch COMPLETE. Replacements applied: 3


In [ ]:
# ---------------------------------------------------------
# Verify complete TF-Slim compatibility patch
# TensorFlow 2.20 / Python 3.13
# ---------------------------------------------------------

verify_patch = subprocess.run(
    [
        TFOD_PYTHON,
        "-c",
        """
import site
from pathlib import Path

# Locate site-packages without importing tf_slim first
site_packages = Path(site.getsitepackages()[0])

p = (
    site_packages
    / "tf_slim"
    / "data"
    / "tfexample_decoder.py"
)

assert p.exists(), f"Decoder not found: {p}"

text = p.read_text(encoding="utf-8")

print("Decoder:", p)

print(
    "TensorFlow import:",
    "import tensorflow as tf" in text
)

print(
    "Old case remaining:",
    "control_flow_ops.case(" in text
)

print(
    "Old cond remaining:",
    "control_flow_ops.cond(" in text
)

print(
    "tf.case present:",
    "tf.case(" in text
)

print(
    "tf.cond present:",
    "tf.cond(" in text
)

# Verify obsolete calls are gone
assert "control_flow_ops.case(" not in text
assert "control_flow_ops.cond(" not in text

# If public replacements exist, TensorFlow must be imported
if "tf.case(" in text or "tf.cond(" in text:
    assert "import tensorflow as tf" in text

# Actual import test
from tf_slim.data import tfexample_decoder

print("TF-Slim decoder import: OK")
"""
    ],
    env=TFOD_RUN_ENV,
    text=True,
    capture_output=True
)

print(verify_patch.stdout)

if verify_patch.stderr:
    print(verify_patch.stderr)

assert verify_patch.returncode == 0, (
    "TF-Slim patch verification failed."
)

print(
    "\nTF-Slim compatibility verification PASSED."
)

Decoder: /content/tfod_env/lib/python3.13/site-packages/tf_slim/data/tfexample_decoder.py
TensorFlow import: True
Old case remaining: False
Old cond remaining: False
tf.case present: True
tf.cond present: True
TF-Slim decoder import: OK


TF-Slim compatibility verification PASSED.


In [ ]:
# ---------------------------------------------------------
# Final pre-training check
# ---------------------------------------------------------

final_check = subprocess.run(
    [
        TFOD_PYTHON,
        "-c",
        (
            "import tensorflow as tf; "
            "from object_detection.builders import model_builder; "
            "print('TensorFlow:', tf.__version__); "
            "print('Estimator:', hasattr(tf.compat.v1, 'estimator')); "
            "print('GPU:', tf.config.list_physical_devices('GPU')); "
            "print('Object Detection imports: OK')"
        )
    ],
    env=TFOD_RUN_ENV,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print(final_check.stdout)

if final_check.returncode != 0:
    raise RuntimeError(
        "Final TFOD environment check failed."
    )

assert "GPU: []" not in final_check.stdout
assert RUNTIME_CONFIG.exists()
assert TRAIN_RECORD.exists()
assert VAL_RECORD.exists()
assert LABEL_MAP_PATH.exists()

print("\nREADY TO TRAIN.")

TensorFlow: 2.20.0
Estimator: False
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Object Detection imports: OK


READY TO TRAIN.


In [ ]:
# ---------------------------------------------------------
# TensorFlow 2.20 compatibility patch
# Make legacy Estimator imports optional in TFOD
# ---------------------------------------------------------

from pathlib import Path

files_to_patch = [
    RESEARCH_DIR / "object_detection" / "inputs.py",
    RESEARCH_DIR / "object_detection" / "model_lib.py",
]

old_import = (
    "from tensorflow.compat.v1 "
    "import estimator as tf_estimator"
)

replacement = """try:
  from tensorflow.compat.v1 import estimator as tf_estimator
except ImportError:
  # TensorFlow >= 2.16 removed Estimator.
  # TF2 training does not require the legacy Estimator training path.
  tf_estimator = None
"""

for path in files_to_patch:

    assert path.exists(), f"Missing file: {path}"

    text = path.read_text(encoding="utf-8")

    if "tf_estimator = None" not in text:

        if old_import in text:
            text = text.replace(
                old_import,
                replacement,
                1
            )
        else:
            print(
                f"No legacy Estimator import found in {path.name}; "
                "leaving unchanged."
            )

    path.write_text(
        text,
        encoding="utf-8"
    )

    patched = path.read_text(
        encoding="utf-8"
    )

    print(
        path.name,
        "patched:",
        "tf_estimator = None" in patched
    )

print(
    "\nLegacy Estimator compatibility patch complete."
)

inputs.py patched: True
model_lib.py patched: True

Legacy Estimator compatibility patch complete.


In [ ]:
# ---------------------------------------------------------
# Verify TF2 training path imports
# ---------------------------------------------------------

import subprocess

check = subprocess.run(
    [
        TFOD_PYTHON,
        "-c",
        (
            "from object_detection import inputs; "
            "from object_detection import model_lib; "
            "from object_detection import model_lib_v2; "
            "print('inputs import: OK'); "
            "print('model_lib import: OK'); "
            "print('model_lib_v2 import: OK'); "
            "print('TF2 training path import: OK')"
        )
    ],
    cwd=str(RESEARCH_DIR),
    env=TFOD_RUN_ENV,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print(check.stdout)

assert check.returncode == 0, (
    "TF2 training-path import still failed."
)

print("\nTF2 training path PASSED.")

inputs import: OK
model_lib import: OK
model_lib_v2 import: OK
TF2 training path import: OK


TF2 training path PASSED.


In [ ]:
# ---------------------------------------------------------
# Train SSD-MobileNetV2 FPNLite 640x640 — V3
# ---------------------------------------------------------

TRAIN_STEPS = 12000

MODEL_MAIN = (
    RESEARCH_DIR
    / "object_detection"
    / "model_main_tf2.py"
)

TRAINING_DIR.mkdir(
    parents=True,
    exist_ok=True
)

command = [
    TFOD_PYTHON,
    str(MODEL_MAIN),
    f"--pipeline_config_path={RUNTIME_CONFIG}",
    f"--model_dir={TRAINING_DIR}",
    f"--num_train_steps={TRAIN_STEPS}",
    "--alsologtostderr",
]

print("Training for", TRAIN_STEPS, "steps")
print("\nTraining command:")
print(" ".join(command))
print("\nStarting training...\n")

process = subprocess.Popen(
    command,
    cwd=str(RESEARCH_DIR),
    env=TFOD_RUN_ENV,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()

print("\nReturn code:", return_code)

if return_code != 0:
    raise RuntimeError(
        "Training failed. The actual TensorFlow/Object Detection "
        "error is printed immediately above."
    )

print("\nTraining completed successfully.")

Training for 12000 steps

Training command:
/content/tfod_env/bin/python /content/models/research/object_detection/model_main_tf2.py --pipeline_config_path=/content/drive/MyDrive/CS PROJECT II/configs/pipeline_runtime.config --model_dir=/content/drive/MyDrive/CS PROJECT II/training_checkpoints/ssd_mobilenet_v2_fpnlite_640_run_v3 --num_train_steps=12000 --alsologtostderr

Starting training...

2026-11-08 22:16:50.748220: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1794176210.749300   18210 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79188 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:00:05.0, compute capability: 8.0
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0',)
I1108 22:16:50.751039 139063873118528 mirrored_strat

## 7. TensorBoard

Run TensorBoard to inspect training loss and evaluation metrics.

```python
%load_ext tensorboard
%tensorboard --logdir "/content/drive/MyDrive/CS PROJECT II/training_checkpoints"
```

Look for:
- total loss
- classification loss
- localization loss
- learning behaviour
- evaluation metrics when evaluation is running

In [ ]:
# ---------------------------------------------------------
# 7.1 Confirm checkpoint artifacts
# ---------------------------------------------------------

checkpoint_files = sorted(
    TRAINING_DIR.glob("ckpt-*")
)

print("Checkpoint artifacts:")

for path in checkpoint_files[-20:]:
    print(path)

if not checkpoint_files:
    raise RuntimeError(
        "No checkpoint artifacts were found."
    )

print("\nCheckpoint output confirmed.")

Checkpoint artifacts:
/content/drive/MyDrive/CS PROJECT II/training_checkpoints/ssd_mobilenet_v2_fpnlite_640_run_v3/ckpt-10.data-00000-of-00001
/content/drive/MyDrive/CS PROJECT II/training_checkpoints/ssd_mobilenet_v2_fpnlite_640_run_v3/ckpt-10.index
/content/drive/MyDrive/CS PROJECT II/training_checkpoints/ssd_mobilenet_v2_fpnlite_640_run_v3/ckpt-11.data-00000-of-00001
/content/drive/MyDrive/CS PROJECT II/training_checkpoints/ssd_mobilenet_v2_fpnlite_640_run_v3/ckpt-11.index
/content/drive/MyDrive/CS PROJECT II/training_checkpoints/ssd_mobilenet_v2_fpnlite_640_run_v3/ckpt-12.data-00000-of-00001
/content/drive/MyDrive/CS PROJECT II/training_checkpoints/ssd_mobilenet_v2_fpnlite_640_run_v3/ckpt-12.index
/content/drive/MyDrive/CS PROJECT II/training_checkpoints/ssd_mobilenet_v2_fpnlite_640_run_v3/ckpt-13.data-00000-of-00001
/content/drive/MyDrive/CS PROJECT II/training_checkpoints/ssd_mobilenet_v2_fpnlite_640_run_v3/ckpt-13.index
/content/drive/MyDrive/CS PROJECT II/training_checkpoints/

## 8. V3.1 Precision-Oriented Fine-Tuning with Validation Early Stopping

In [ ]:
# ---------------------------------------------------------
# V3.1 paths
# Precision-oriented fine-tuning from best V3 checkpoint
# ---------------------------------------------------------

from pathlib import Path
import shutil
import re

V3_DIR = (
    PROJECT_DIR
    / "training_checkpoints"
    / "ssd_mobilenet_v2_fpnlite_640_run_v3"
)

V3_1_DIR = (
    PROJECT_DIR
    / "training_checkpoints"
    / "ssd_mobilenet_v2_fpnlite_640_run_v3_1"
)

BEST_V3_1_DIR = (
    PROJECT_DIR
    / "training_checkpoints"
    / "ssd_mobilenet_v2_fpnlite_640_run_v3_1_best"
)

V3_1_CONFIG = (
    PROJECT_DIR
    / "configs"
    / "pipeline_v3_1.config"
)

# Best V3 checkpoint identified by Notebook 06
V3_START_CHECKPOINT = (
    V3_DIR
    / "ckpt-13"
)

assert (
    Path(str(V3_START_CHECKPOINT) + ".index")
    .exists()
), (
    f"Missing V3 checkpoint: {V3_START_CHECKPOINT}"
)

print("V3 starting checkpoint:", V3_START_CHECKPOINT)
print("V3.1 training directory:", V3_1_DIR)
print("V3.1 config:", V3_1_CONFIG)

V3 starting checkpoint: /content/drive/MyDrive/CS PROJECT II/training_checkpoints/ssd_mobilenet_v2_fpnlite_640_run_v3/ckpt-13
V3.1 training directory: /content/drive/MyDrive/CS PROJECT II/training_checkpoints/ssd_mobilenet_v2_fpnlite_640_run_v3_1
V3.1 config: /content/drive/MyDrive/CS PROJECT II/configs/pipeline_v3_1.config


In [ ]:
# ---------------------------------------------------------
# V3.1 prerequisite:
# Restore TensorFlow 2.20 compatibility patches
# after a fresh Colab runtime
# ---------------------------------------------------------

from pathlib import Path
import os
import subprocess


# =========================================================
# 1. Ensure TFOD subprocess environment is correct
# =========================================================

TFOD_RUN_ENV["PYTHONPATH"] = (
    f"{RESEARCH_DIR}:{SLIM_DIR}"
)

TFOD_RUN_ENV["MPLBACKEND"] = "Agg"

TFOD_RUN_ENV["TF_USE_LEGACY_KERAS"] = "1"


# =========================================================
# 2. Make legacy Estimator imports optional
# =========================================================

ESTIMATOR_FILES = [
    (
        RESEARCH_DIR
        / "object_detection"
        / "inputs.py"
    ),
    (
        RESEARCH_DIR
        / "object_detection"
        / "model_lib.py"
    ),
]

OLD_ESTIMATOR_IMPORT = (
    "from tensorflow.compat.v1 "
    "import estimator as tf_estimator"
)

ESTIMATOR_REPLACEMENT = """try:
  from tensorflow.compat.v1 import estimator as tf_estimator
except ImportError:
  # TensorFlow >= 2.16 removed Estimator.
  # TF2 Object Detection training/evaluation uses model_lib_v2.
  tf_estimator = None
"""


for path in ESTIMATOR_FILES:

    assert path.exists(), (
        f"Missing TFOD file: {path}"
    )

    text = path.read_text(
        encoding="utf-8"
    )

    if "tf_estimator = None" not in text:

        if OLD_ESTIMATOR_IMPORT in text:

            text = text.replace(
                OLD_ESTIMATOR_IMPORT,
                ESTIMATOR_REPLACEMENT,
                1
            )

            path.write_text(
                text,
                encoding="utf-8"
            )

            print(
                f"Patched Estimator import: "
                f"{path.name}"
            )

        else:

            print(
                f"No Estimator patch required: "
                f"{path.name}"
            )

    else:

        print(
            f"Estimator patch already present: "
            f"{path.name}"
        )


# =========================================================
# 3. Make unused DeepMAC / tensorflow_io dependency optional
# =========================================================

MODEL_BUILDER = (
    RESEARCH_DIR
    / "object_detection"
    / "builders"
    / "model_builder.py"
)

assert MODEL_BUILDER.exists(), (
    f"Missing model_builder.py: "
    f"{MODEL_BUILDER}"
)

builder_text = MODEL_BUILDER.read_text(
    encoding="utf-8"
)

OLD_DEEPMAC_IMPORT = (
    "from object_detection.meta_architectures "
    "import deepmac_meta_arch"
)

DEEPMAC_REPLACEMENT = """try:
  from object_detection.meta_architectures import deepmac_meta_arch
except ModuleNotFoundError as exc:
  if exc.name == 'tensorflow_io':
    deepmac_meta_arch = None
  else:
    raise
"""


if "deepmac_meta_arch = None" not in builder_text:

    if OLD_DEEPMAC_IMPORT in builder_text:

        builder_text = builder_text.replace(
            OLD_DEEPMAC_IMPORT,
            DEEPMAC_REPLACEMENT,
            1
        )

        MODEL_BUILDER.write_text(
            builder_text,
            encoding="utf-8"
        )

        print(
            "Patched optional DeepMAC dependency."
        )

else:

    print(
        "DeepMAC patch already present."
    )


# =========================================================
# 4. TF-Slim TensorFlow 2.20 compatibility
# =========================================================

site_check = subprocess.run(
    [
        TFOD_PYTHON,
        "-c",
        (
            "import site; "
            "print(site.getsitepackages()[0])"
        ),
    ],
    env=TFOD_RUN_ENV,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

assert site_check.returncode == 0, (
    site_check.stdout
)

TFOD_SITE = Path(
    site_check.stdout.strip()
)

TFSLIM_DECODER = (
    TFOD_SITE
    / "tf_slim"
    / "data"
    / "tfexample_decoder.py"
)

assert TFSLIM_DECODER.exists(), (
    f"Missing TF-Slim decoder: "
    f"{TFSLIM_DECODER}"
)

slim_text = TFSLIM_DECODER.read_text(
    encoding="utf-8"
)

needs_slim_patch = (
    "control_flow_ops.case(" in slim_text
    or
    "control_flow_ops.cond(" in slim_text
)


if needs_slim_patch:

    if "import tensorflow as tf" not in slim_text:

        lines = slim_text.splitlines()

        future_indexes = [
            i
            for i, line in enumerate(lines)
            if line.strip().startswith(
                "from __future__ import"
            )
        ]

        insert_at = (
            max(future_indexes) + 1
            if future_indexes
            else 0
        )

        lines.insert(
            insert_at,
            "import tensorflow as tf"
        )

        slim_text = (
            "\n".join(lines)
            + "\n"
        )

    slim_text = slim_text.replace(
        "control_flow_ops.case(",
        "tf.case("
    )

    slim_text = slim_text.replace(
        "control_flow_ops.cond(",
        "tf.cond("
    )

    TFSLIM_DECODER.write_text(
        slim_text,
        encoding="utf-8"
    )

    print(
        "Patched TF-Slim control-flow APIs."
    )

else:

    print(
        "TF-Slim patch already present "
        "or not required."
    )


# =========================================================
# 5. Verify ACTUAL V3.1 training path
# =========================================================

print(
    "\nVerifying TF2 training path..."
)

verify = subprocess.run(
    [
        TFOD_PYTHON,
        "-c",
        """
import tensorflow as tf
import tf_keras

from object_detection import inputs
from object_detection import model_lib
from object_detection import model_lib_v2
from object_detection.builders import model_builder
from object_detection.utils import config_util

print("TensorFlow:", tf.__version__)
print("TF-Keras:", tf_keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

print("inputs import: OK")
print("model_lib import: OK")
print("model_lib_v2 import: OK")
print("model_builder import: OK")

print("V3.1 TF2 training path: READY")
"""
    ],
    cwd=str(RESEARCH_DIR),
    env=TFOD_RUN_ENV,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print(
    verify.stdout
)

assert verify.returncode == 0, (
    "TFOD compatibility verification failed."
)

assert (
    "V3.1 TF2 training path: READY"
    in verify.stdout
)

assert "GPU: []" not in verify.stdout, (
    "GPU is not visible to TensorFlow."
)

print(
    "\nV3.1 COMPATIBILITY PATCH COMPLETE."
)

Estimator patch already present: inputs.py
Estimator patch already present: model_lib.py
DeepMAC patch already present.
TF-Slim patch already present or not required.

Verifying TF2 training path...
TensorFlow: 2.20.0
TF-Keras: 2.20.1
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
inputs import: OK
model_lib import: OK
model_lib_v2 import: OK
model_builder import: OK
V3.1 TF2 training path: READY


V3.1 COMPATIBILITY PATCH COMPLETE.


In [ ]:
# ---------------------------------------------------------
# 8.3 Clear failed V3.1 experiment
# ---------------------------------------------------------

from pathlib import Path
import shutil

BROKEN_DIRS = [
    V3_1_DIR,
    BEST_V3_1_DIR,
]

for path in BROKEN_DIRS:

    if path.exists():

        shutil.rmtree(
            path
        )

        print(
            "Removed failed V3.1 directory:",
            path
        )


print(
    "\nFailed V3.1 experiment cleared."
)


Failed V3.1 experiment cleared.


In [ ]:
# ---------------------------------------------------------
# 8.4 Create corrected V3.1 configuration
# Full V3 detector will be restored separately
# ---------------------------------------------------------

import subprocess
import shutil

shutil.copy2(
    RUNTIME_CONFIG,
    V3_1_CONFIG
)


patch_v31 = f"""
from google.protobuf import text_format
from object_detection.protos import pipeline_pb2

config_path = {str(V3_1_CONFIG)!r}
train_record = {str(TRAIN_RECORD)!r}
val_record = {str(VAL_RECORD)!r}
label_map = {str(LABEL_MAP_PATH)!r}

config = pipeline_pb2.TrainEvalPipelineConfig()

with open(config_path, "r", encoding="utf-8") as f:
    text_format.Merge(
        f.read(),
        config
    )

# ---------------------------------------------------------
# Preserve V3 architecture
# ---------------------------------------------------------

config.model.ssd.num_classes = 3

config.model.ssd.image_resizer.fixed_shape_resizer.height = 640
config.model.ssd.image_resizer.fixed_shape_resizer.width = 640

# Stabilize small-batch fine-tuning
config.model.ssd.freeze_batchnorm = True

# ---------------------------------------------------------
# IMPORTANT:
# Do NOT initialize using TFOD fine_tune_checkpoint.
# The complete V3 detector will be restored separately.
# ---------------------------------------------------------

config.train_config.fine_tune_checkpoint = ""

config.train_config.batch_size = 4

# ---------------------------------------------------------
# Gentle V3.1 learning-rate schedule
# ---------------------------------------------------------

lr = (
    config.train_config
    .optimizer
    .momentum_optimizer
    .learning_rate
    .cosine_decay_learning_rate
)

lr.learning_rate_base = 0.00015
lr.warmup_learning_rate = 0.00003
lr.warmup_steps = 100
lr.total_steps = 2500

# ---------------------------------------------------------
# Controlled augmentations
# ---------------------------------------------------------

config.train_config.ClearField(
    "data_augmentation_options"
)

step = (
    config.train_config
    .data_augmentation_options
    .add()
)

step.random_horizontal_flip.SetInParent()


step = (
    config.train_config
    .data_augmentation_options
    .add()
)

step.random_adjust_brightness.max_delta = 0.05


step = (
    config.train_config
    .data_augmentation_options
    .add()
)

step.random_adjust_contrast.min_delta = 0.95
step.random_adjust_contrast.max_delta = 1.05


# ---------------------------------------------------------
# Training / validation records
# ---------------------------------------------------------

config.train_input_reader.label_map_path = (
    label_map
)

config.train_input_reader.tf_record_input_reader.input_path[:] = [
    train_record
]


config.eval_input_reader[0].label_map_path = (
    label_map
)

config.eval_input_reader[0].tf_record_input_reader.input_path[:] = [
    val_record
]

config.eval_input_reader[0].shuffle = False
config.eval_input_reader[0].num_epochs = 1


with open(
    config_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        text_format.MessageToString(
            config
        )
    )


print(
    "Architecture:",
    config.model.ssd.feature_extractor.type
)

print(
    "Resolution:",
    config.model.ssd.image_resizer.fixed_shape_resizer.height,
    "x",
    config.model.ssd.image_resizer.fixed_shape_resizer.width
)

print(
    "Freeze batchnorm:",
    config.model.ssd.freeze_batchnorm
)

print(
    "Batch size:",
    config.train_config.batch_size
)

print(
    "Fine tune checkpoint:",
    repr(
        config.train_config.fine_tune_checkpoint
    )
)

print(
    "Base LR:",
    lr.learning_rate_base
)

print(
    "Warmup LR:",
    lr.warmup_learning_rate
)

print(
    "Warmup steps:",
    lr.warmup_steps
)

print(
    "Maximum steps:",
    lr.total_steps
)

print(
    "Augmentations:",
    [
        step.WhichOneof(
            "preprocessing_step"
        )
        for step
        in config.train_config.data_augmentation_options
    ]
)
"""


result = subprocess.run(
    [
        TFOD_PYTHON,
        "-c",
        patch_v31
    ],
    env=TFOD_RUN_ENV,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print(
    result.stdout
)

assert result.returncode == 0, (
    "V3.1 configuration creation failed."
)

print(
    "\nCorrected V3.1 configuration READY."
)

Architecture: ssd_mobilenet_v2_fpn_keras
Resolution: 640 x 640
Freeze batchnorm: True
Batch size: 4
Fine tune checkpoint: ''
Base LR: 0.0001500000071246177
Warmup LR: 2.9999999242136255e-05
Warmup steps: 100
Maximum steps: 2500
Augmentations: ['random_horizontal_flip', 'random_adjust_brightness', 'random_adjust_contrast']


Corrected V3.1 configuration READY.


In [ ]:
# ---------------------------------------------------------
# 8.5 Bootstrap V3.1 from COMPLETE V3 detector
# ---------------------------------------------------------

import subprocess
import shutil
from pathlib import Path


V3_CHECKPOINT = (
    V3_DIR
    / "ckpt-13"
)


assert Path(
    str(V3_CHECKPOINT)
    + ".index"
).exists(), (
    f"Missing V3 checkpoint: "
    f"{V3_CHECKPOINT}"
)


V3_1_DIR.mkdir(
    parents=True,
    exist_ok=True
)


bootstrap_script = f"""
import tensorflow as tf

from object_detection.utils import config_util
from object_detection.builders import model_builder

config_path = {str(V3_1_CONFIG)!r}
source_checkpoint = {str(V3_CHECKPOINT)!r}
output_dir = {str(V3_1_DIR)!r}


configs = (
    config_util
    .get_configs_from_pipeline_file(
        config_path
    )
)

model_config = configs["model"]


model = model_builder.build(
    model_config=model_config,
    is_training=True
)


# Build model variables
dummy = tf.zeros(
    [1, 640, 640, 3],
    dtype=tf.float32
)

preprocessed, shapes = (
    model.preprocess(
        dummy
    )
)

prediction_dict = (
    model.predict(
        preprocessed,
        shapes
    )
)

_ = model.postprocess(
    prediction_dict,
    shapes
)


print(
    "Model variables built:",
    len(model.variables)
)


# Restore complete V3 detector
source_ckpt = tf.train.Checkpoint(
    model=model
)

status = source_ckpt.restore(
    source_checkpoint
)

status.assert_existing_objects_matched()


print(
    "Full V3 detector restore: OK"
)


# Reset training step for new experiment
step = tf.Variable(
    0,
    trainable=False,
    dtype=tf.int64,
    name="global_step"
)


new_ckpt = tf.train.Checkpoint(
    step=step,
    model=model
)


manager = tf.train.CheckpointManager(
    new_ckpt,
    output_dir,
    max_to_keep=7
)


saved = manager.save()


print(
    "Fresh V3.1 bootstrap checkpoint:",
    saved
)

print(
    "Bootstrap step:",
    int(step.numpy())
)
"""


bootstrap = subprocess.run(
    [
        TFOD_PYTHON,
        "-c",
        bootstrap_script
    ],
    cwd=str(RESEARCH_DIR),
    env=TFOD_RUN_ENV,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)


print(
    bootstrap.stdout
)


assert bootstrap.returncode == 0, (
    "Full V3 checkpoint bootstrap failed."
)

assert (
    "Full V3 detector restore: OK"
    in bootstrap.stdout
)


print(
    "\nV3.1 FULL-MODEL BOOTSTRAP PASSED."
)

2026-11-10 07:50:53.156809: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1794297053.157980   20793 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38477 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:00:04.0, compute capability: 8.0
Model variables built: 384
Full V3 detector restore: OK
Fresh V3.1 bootstrap checkpoint: /content/drive/MyDrive/CS PROJECT II/training_checkpoints/ssd_mobilenet_v2_fpnlite_640_run_v3_1/ckpt-1
Bootstrap step: 0


V3.1 FULL-MODEL BOOTSTRAP PASSED.


In [ ]:
# ---------------------------------------------------------
# 8.6 Verify restored V3 BEFORE fine-tuning
# ---------------------------------------------------------

import subprocess
import shutil
import re
from pathlib import Path


MODEL_MAIN = (
    RESEARCH_DIR
    / "object_detection"
    / "model_main_tf2.py"
)


PRETRAIN_EVAL_DIR = Path(
    "/content/v3_1_pretrain_eval"
)


if PRETRAIN_EVAL_DIR.exists():

    shutil.rmtree(
        PRETRAIN_EVAL_DIR
    )


PRETRAIN_EVAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


command = [
    TFOD_PYTHON,
    str(MODEL_MAIN),

    (
        "--pipeline_config_path="
        + str(V3_1_CONFIG)
    ),

    (
        "--model_dir="
        + str(PRETRAIN_EVAL_DIR)
    ),

    (
        "--checkpoint_dir="
        + str(V3_1_DIR)
    ),

    "--eval_timeout=1",
    "--alsologtostderr",
]


result = subprocess.run(
    command,
    cwd=str(RESEARCH_DIR),
    env=TFOD_RUN_ENV,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)


print(
    result.stdout
)


matches = re.findall(
    (
        r"DetectionBoxes_Precision/"
        r"mAP:\s+([0-9.]+)"
    ),
    result.stdout
)


assert matches, (
    "Could not extract restored "
    "V3 validation mAP."
)


RESTORED_MAP = float(
    matches[-1]
)


print(
    "\nRestored V3 validation mAP:",
    RESTORED_MAP
)


EXPECTED_V3_MAP = 0.117777
RESTORE_TOLERANCE = 0.015

assert abs(
    RESTORED_MAP - EXPECTED_V3_MAP
) <= RESTORE_TOLERANCE, (
    f"STOP. Restored validation mAP = {RESTORED_MAP:.6f}, "
    f"but expected approximately {EXPECTED_V3_MAP:.6f}. "
    "The complete V3 detector was not reproduced reliably. "
    "Do NOT start V3.1 training."
)


print(
    "\nFULL V3 RESTORE VERIFIED."
)

print(
    "Safe to begin V3.1 fine-tuning."
)

W1110 07:51:43.276639 135634434523456 model_lib_v2.py:1089] Forced number of epochs for all eval validations to be 1.
INFO:tensorflow:Maybe overwriting sample_1_of_n_eval_examples: None
I1110 07:51:43.276844 135634434523456 config_util.py:552] Maybe overwriting sample_1_of_n_eval_examples: None
INFO:tensorflow:Maybe overwriting use_bfloat16: False
I1110 07:51:43.276931 135634434523456 config_util.py:552] Maybe overwriting use_bfloat16: False
INFO:tensorflow:Maybe overwriting eval_num_epochs: 1
I1110 07:51:43.277026 135634434523456 config_util.py:552] Maybe overwriting eval_num_epochs: 1
W1110 07:51:43.277141 135634434523456 model_lib_v2.py:1106] Expected number of evaluation epochs is 1, but instead encountered `eval_on_train_input_config.num_epochs` = 0. Overwriting `num_epochs` to 1.
2026-11-10 07:51:43.525895: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config 

In [ ]:
# ---------------------------------------------------------
# V3.1 staged training with REAL validation early stopping
# ---------------------------------------------------------

import subprocess
import shutil
import re
import pandas as pd
from pathlib import Path

MODEL_MAIN = (
    RESEARCH_DIR
    / "object_detection"
    / "model_main_tf2.py"
)

MAX_STEPS = 2500
EVAL_INTERVAL = 500

PATIENCE = 2
MIN_DELTA = 0.001

V3_1_DIR.mkdir(
    parents=True,
    exist_ok=True
)

if BEST_V3_1_DIR.exists():
    shutil.rmtree(BEST_V3_1_DIR)

BEST_V3_1_DIR.mkdir(
    parents=True,
    exist_ok=True
)




metric_patterns = {
    "mAP":
        r"DetectionBoxes_Precision/mAP:\s+([0-9.]+)",

    "AP50":
        r"DetectionBoxes_Precision/mAP@\.50IOU:\s+([0-9.]+)",

    "AP75":
        r"DetectionBoxes_Precision/mAP@\.75IOU:\s+([0-9.]+)",

    "AP_small":
        r"DetectionBoxes_Precision/mAP \(small\):\s+([0-9.]+)",

    "AP_medium":
        r"DetectionBoxes_Precision/mAP \(medium\):\s+([0-9.]+)",

    "AP_large":
        r"DetectionBoxes_Precision/mAP \(large\):\s+([0-9.]+)",

    "AR100":
        r"DetectionBoxes_Recall/AR@100:\s+([0-9.]+)",

    "total_loss":
        r"Loss/total_loss:\s+([0-9.]+)",
}


def latest_checkpoint_prefix(directory):
    indexes = list(
        directory.glob("ckpt-*.index")
    )

    assert indexes, (
        f"No checkpoint found in {directory}"
    )

    indexes.sort(
        key=lambda p: int(
            re.search(
                r"ckpt-(\d+)",
                p.name
            ).group(1)
        )
    )

    return indexes[-1].with_suffix("")


def copy_checkpoint(prefix, destination):

    destination.mkdir(
        parents=True,
        exist_ok=True
    )

    for old in destination.glob("ckpt-*"):
        old.unlink()

    checkpoint_number = prefix.name

    for src in prefix.parent.glob(
        checkpoint_number + ".*"
    ):
        shutil.copy2(
            src,
            destination / src.name
        )

    (
        destination
        / "checkpoint"
    ).write_text(
        (
            f'model_checkpoint_path: "{checkpoint_number}"\n'
            f'all_model_checkpoint_paths: "{checkpoint_number}"\n'
        ),
        encoding="utf-8"
    )

  # Preserve the restored V3 checkpoint as the fallback best model.
baseline_prefix = latest_checkpoint_prefix(
    V3_1_DIR
)

copy_checkpoint(
    baseline_prefix,
    BEST_V3_1_DIR
)


def evaluate_latest_validation():

    EVAL_LOG_DIR = Path(
        "/content/v3_1_validation_eval"
    )

    if EVAL_LOG_DIR.exists():
        shutil.rmtree(
            EVAL_LOG_DIR
        )

    EVAL_LOG_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    command = [
        TFOD_PYTHON,
        str(MODEL_MAIN),

        f"--pipeline_config_path={V3_1_CONFIG}",
        f"--model_dir={EVAL_LOG_DIR}",
        f"--checkpoint_dir={V3_1_DIR}",

        "--eval_timeout=1",
        "--alsologtostderr",
    ]

    process = subprocess.Popen(
        command,
        cwd=str(RESEARCH_DIR),
        env=TFOD_RUN_ENV,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1
    )

    lines = []

    for line in process.stdout:

        lines.append(line)

        if (
            "DetectionBoxes_Precision/mAP:" in line
            or
            "DetectionBoxes_Precision/mAP@.50IOU:" in line
            or
            "DetectionBoxes_Precision/mAP@.75IOU:" in line
            or
            "DetectionBoxes_Precision/mAP (small):" in line
            or
            "DetectionBoxes_Precision/mAP (medium):" in line
            or
            "DetectionBoxes_Precision/mAP (large):" in line
            or
            "DetectionBoxes_Recall/AR@100:" in line
            or
            "Loss/total_loss:" in line
        ):
            print(line, end="")

        # Last metric we need
        if "Loss/total_loss:" in line:
            break

    if process.poll() is None:

        process.terminate()

        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()

    output = "".join(lines)

    metrics = {}

    for name, pattern in metric_patterns.items():

        matches = re.findall(
            pattern,
            output
        )

        metrics[name] = (
            float(matches[-1])
            if matches
            else None
        )

    if metrics["mAP"] is None:
        print(output)
        raise RuntimeError(
            "Validation metrics could not be extracted."
        )

    return metrics


# Start from the correctly restored V3 baseline.
history = [
    {
        "step": 0,
        "mAP": RESTORED_MAP
    }
]

best_map = RESTORED_MAP
bad_rounds = 0
best_step = 0


for target_step in range(
    EVAL_INTERVAL,
    MAX_STEPS + 1,
    EVAL_INTERVAL
):

    print(
        "\n"
        + "=" * 72
    )

    print(
        f"TRAINING V3.1 TO STEP {target_step}"
    )

    print(
        "=" * 72
    )


    train_command = [
        TFOD_PYTHON,
        str(MODEL_MAIN),

        f"--pipeline_config_path={V3_1_CONFIG}",
        f"--model_dir={V3_1_DIR}",
        f"--num_train_steps={target_step}",

        f"--checkpoint_every_n={EVAL_INTERVAL}",

        "--alsologtostderr",
    ]


    train_result = subprocess.run(
        train_command,
        cwd=str(RESEARCH_DIR),
        env=TFOD_RUN_ENV,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )


    # Print only tail to avoid enormous notebook output
    tail = train_result.stdout.splitlines()[-40:]

    print(
        "\n".join(tail)
    )


    if train_result.returncode != 0:

        raise RuntimeError(
            f"Training failed before step {target_step}."
        )


    print(
        f"\nVALIDATING STEP {target_step}"
    )


    metrics = evaluate_latest_validation()

    row = {
        "step": target_step,
        **metrics
    }

    history.append(row)

    current_map = metrics["mAP"]


    print(
        "\nValidation metrics:",
        row
    )


    if current_map > best_map + MIN_DELTA:

        best_map = current_map
        best_step = target_step

        bad_rounds = 0

        best_prefix = (
            latest_checkpoint_prefix(
                V3_1_DIR
            )
        )

        copy_checkpoint(
            best_prefix,
            BEST_V3_1_DIR
        )

        print(
            f"\nNEW BEST CHECKPOINT "
            f"| step={best_step} "
            f"| mAP={best_map:.6f}"
        )

    else:

        bad_rounds += 1

        print(
            f"\nNo meaningful validation improvement. "
            f"Patience {bad_rounds}/{PATIENCE}"
        )


    if bad_rounds >= PATIENCE:

        print(
            "\nEARLY STOPPING TRIGGERED."
        )

        break


v31_history_df = pd.DataFrame(
    history
)

display(
    v31_history_df
)

V3_1_HISTORY_CSV = (
    PROJECT_DIR
    / "report_figures"
    / "v3_1_validation_history.csv"
)

V3_1_HISTORY_CSV.parent.mkdir(
    parents=True,
    exist_ok=True
)

v31_history_df.to_csv(
    V3_1_HISTORY_CSV,
    index=False
)

print(
    "\nBest V3.1 validation step:",
    best_step
)

print(
    "Best V3.1 validation mAP:",
    best_map
)

print(
    "Best checkpoint directory:",
    BEST_V3_1_DIR
)

print(
    "\nREAL VALIDATION-BASED EARLY STOPPING COMPLETE."
)


TRAINING V3.1 TO STEP 500
 'Loss/total_loss': np.float32(0.5102016),
 'learning_rate': np.float32(0.00014429099)}
I1110 07:58:37.360931 135940022010176 model_lib_v2.py:708] {'Loss/classification_loss': np.float32(0.22973149),
 'Loss/localization_loss': np.float32(0.12905501),
 'Loss/regularization_loss': np.float32(0.15141511),
 'Loss/total_loss': np.float32(0.5102016),
 'learning_rate': np.float32(0.00014429099)}
INFO:tensorflow:Step 500 per-step time 0.078s
I1110 07:58:45.140695 135940022010176 model_lib_v2.py:705] Step 500 per-step time 0.078s
INFO:tensorflow:{'Loss/classification_loss': np.float32(0.31260267),
 'Loss/localization_loss': np.float32(0.46626157),
 'Loss/regularization_loss': np.float32(0.15141647),
 'Loss/total_loss': np.float32(0.93028075),
 'learning_rate': np.float32(0.00013995191)}
I1110 07:58:45.140980 135940022010176 model_lib_v2.py:708] {'Loss/classification_loss': np.float32(0.31260267),
 'Loss/localization_loss': np.float32(0.46626157),
 'Loss/regularization

,step,mAP,AP50,AP75,AP_small,AP_medium,AP_large,AR100,total_loss
0,0,0.117300,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,500,0.117994,0.279317,0.080306,0.081473,0.139471,0.148728,0.368338,1.078422
2,1000,0.131116,0.305394,0.078018,0.070947,0.152072,0.162765,0.377391,1.090649
3,1500,0.129323,0.292995,0.080113,0.081924,0.150934,0.159519,0.370843,1.055042
4,2000,0.130349,0.296873,0.091623,0.086660,0.153919,0.161948,0.393031,1.056448



Best V3.1 validation step: 1000
Best V3.1 validation mAP: 0.131116
Best checkpoint directory: /content/drive/MyDrive/CS PROJECT II/training_checkpoints/ssd_mobilenet_v2_fpnlite_640_run_v3_1_best

REAL VALIDATION-BASED EARLY STOPPING COMPLETE.


### Application integration checkpoint

Notebook 05 does not need the Flask application.

The application will only need the **exported inference model** produced by Notebook 07.

Before deployment, the application team must confirm:
1. which exported format it will consume (`SavedModel` and/or `.tflite`);
2. expected model input size;
3. class ID ordering;
4. output tensor names/shapes if using TFLite;
5. model artifact path expected by `backend/app/services/inference.py`.

Do not place large model weights into Git unless deliberately required.